In [48]:
import pandas as pd
from commons.config import *
from main import process_batdongsan_df, process_onehousing_df
from Batdongsan.address_standardizer import AddressStandardizer
import re
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
# bds = pd.read_csv(DETAILS_CSV_PATH['Batdongsan'])
# oh = pd.read_csv(DETAILS_CSV_PATH['Onehousing'])

standardizer = AddressStandardizer(
            PROVINCES_SQL_FILE, DISTRICTS_SQL_FILE, 
            WARDS_SQL_FILE, STREETS_SQL_FILE
        )

bds_clean = process_batdongsan_df(standardizer=standardizer)
oh_clean = process_onehousing_df()

Ground truth file not found.
Dropped 14 duplicated rows for Batdongsan.
Dropped 882 NaN rows for Batdongsan.
Final number of rows for Batdongsan: 104
[OneHousing] Starting data cleaning...
[OneHousing] Data cleaning completed.
Dropped 1 duplicated rows for Onehousing.
Final number of rows for Onehousing: 999


In [5]:
oh_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Tỉnh/Thành phố                        999 non-null    object 
 1   Thành phố/Quận/Huyện/Thị xã           999 non-null    object 
 2   Xã/Phường/Thị trấn                    0 non-null      float64
 3   Đường phố                             999 non-null    object 
 4   Chi tiết                              999 non-null    object 
 5   Nguồn thông tin                       999 non-null    object 
 6   Tình trạng giao dịch                  999 non-null    object 
 7   Thời điểm giao dịch/rao bán           0 non-null      float64
 8   Thông tin liên hệ                     999 non-null    object 
 9   Giá rao bán/giao dịch                 999 non-null    float64
 10  Giá ước tính                          999 non-null    float64
 11  Loại đơn giá (đ/m2 hoặc 

In [9]:
oh_clean.iloc[0]

Tỉnh/Thành phố                                                           Thành phố Hà Nội
Thành phố/Quận/Huyện/Thị xã                                               Huyện Thanh Trì
Xã/Phường/Thị trấn                                                                    NaN
Đường phố                                                                       Tân Triều
Chi tiết                                                                          Mặt ngõ
Nguồn thông tin                         https://onehousing.vn/bds/Nha-mat-ngo-cach-Tan...
Tình trạng giao dịch                                                       Đang giao dịch
Thời điểm giao dịch/rao bán                                                           NaN
Thông tin liên hệ                                                                        
Giá rao bán/giao dịch                                                        5850000000.0
Giá ước tính                                                                 5733000000.0
Loại đơn g

# Inspect Onehousing

## Code address standardizer

In [12]:
import argparse
import sqlite3

class AddressStandardizer:
    def __init__(self, provinces_sql_path: str, districts_sql_path: str, wards_sql_path: str, streets_sql_path: str):
        self.reverse_province_map = {}
        self.reverse_district_map = {}
        self.provinces_sql_path = provinces_sql_path
        self.districts_sql_path = districts_sql_path
        self.wards_sql_path = wards_sql_path
        self.streets_sql_path = streets_sql_path
        self._load_data()

    def _load_data(self):
        try:
            conn = sqlite3.connect(":memory:")
            conn.execute("CREATE TABLE provinces (name TEXT, code TEXT, status TEXT);")
            conn.execute("CREATE TABLE districts (name TEXT, code TEXT, province_code TEXT, status TEXT);")
            conn.execute("CREATE TABLE wards (name TEXT, code TEXT, district_code TEXT, status TEXT);")
            conn.execute("CREATE TABLE streets (name TEXT, code TEXT, district_code TEXT, status TEXT);")

            with open(self.provinces_sql_path, "r", encoding="utf-8") as f:
                conn.executescript(f.read())

            with open(self.districts_sql_path, "r", encoding="utf-8") as f:
                dis_cleaned = f.read().replace("\\'", "''")
                conn.executescript(dis_cleaned)

            with open(self.wards_sql_path, "r", encoding="utf-8") as f:
                ward_cleaned = f.read().replace("\\'", "''")
                conn.executescript(ward_cleaned)

            with open(self.streets_sql_path, "r", encoding="utf-8") as f:
                street_cleaned = f.read().replace("'\\'", "''")
                street_cleaned = f.read().replace("'\'", "''")
                conn.executescript(street_cleaned)

            provinces_df = pd.read_sql_query("SELECT * FROM provinces", conn)

            districts_df = pd.read_sql_query("""
                SELECT d.name AS district_name, p.name AS province_name
                FROM districts d
                JOIN provinces p ON d.province_code = p.code
                """, conn)
            
            wards_df = pd.read_sql_query("""
                SELECT w.name AS ward_name,
                    d.name AS district_name,
                    p.name AS province_name
                FROM wards w
                JOIN districts d ON w.district_code = d.code
                JOIN provinces p ON d.province_code = p.code
            """, conn)

        except (FileNotFoundError, sqlite3.Error) as e:
            print(f"Error: Could not load administrative data. Details: {e}")
            raise
        finally:
            if 'conn' in locals():
                conn.close()

        # Build reverse lookup maps
        self.reverse_province_map = {
            prov.replace("Thành phố ", "").replace("Tỉnh ", ""): prov
            for prov in provinces_df['name'].unique()
        }
        self.reverse_province_map['Bà Rịa Vũng Tàu'] = 'Tỉnh Bà Rịa - Vũng Tàu'

        self.reverse_district = {}
        for province in districts_df['province_name'].unique():
            self.reverse_district[province] = {}
            for district_name in districts_df[districts_df['province_name'] == province]['district_name'].unique():
                district_name_strip = district_name.replace('Thành phố ', '').replace('Thành Phố ', '').replace('Quận ', '').replace('Huyện ', '').replace('Thị xã ', '').replace('Thị Xã ', '').strip()
                self.reverse_district[province][district_name_strip] = district_name
        self.reverse_district['Tỉnh Bà Rịa - Vũng Tàu']['Long Đất'] = 'Huyện Long Đất'
        self.reverse_district['Thành phố Hồ Chí Minh']['Quận 2'] = 'Thành phố Thủ Đức'
        self.reverse_district['Thành phố Hồ Chí Minh']['Quận 9'] = 'Thành phố Thủ Đức'

        self.reverse_ward = {}
        for province in self.reverse_district.keys():
            self.reverse_ward[province] = {}
            for district in self.reverse_district[province].values():
                self.reverse_ward[province][district] = {}
                for ward in wards_df[wards_df['district_name'] == district]['ward_name'].unique():
                    ward_name_strip = normalize('NFC', ward.replace('Xã ', '').replace('Phường ', '').replace('Thị trấn ', '').replace('Thị Trấn ', '').strip())
                    self.reverse_ward[province][district][ward_name_strip] = ward
        
        self.districts = districts_df
        self.wards = wards_df

    def standardize_province(self, province_name):
        if not isinstance(province_name, str):
            return province_name
        province_name = province_name.replace('.', '')
        province_name = province_name.replace(',', '')
        province_name = province_name.replace('?', '')
        province_name = province_name.replace('!', '')
        return self.reverse_province_map.get(province_name, province_name)

    def standardize_district(self, row):
            prefix = ['Thành phố', 'Thành Phố', 'Quận', 'Huyện', 'Thị xã', 'Thị Xã', 'Đảo']
            district_value = row['Thành phố/Quận/Huyện/Thị xã']

            if isinstance(district_value, str):
                if district_value == 'Quận 2' or district_value == 'Quận 9':
                    return 'Thành phố Thủ Đức'
                for pre in prefix:
                    if district_value.startswith(pre):
                        return district_value
                province = row['Tỉnh/Thành phố']
                if district_value in self.reverse_district[province].keys():
                    return self.reverse_district[province][district_value]
                for dis in self.reverse_district[province].keys():
                    similarity = fuzz.ratio(district_value, dis)
                    if similarity >= 66:
                        return self.reverse_district[province][dis]
                return None
            return None

    def standardize_ward(self, row):
        ward_value = row['Xã/Phường/Thị trấn']

        def matching(ward_value, district_value, province_value):
            # Function to match values with its corresponding prefixes
            if ward_value in self.reverse_ward[province_value][district_value].keys():
                return self.reverse_ward[province_value][district_value][ward_value]
            for ward in self.reverse_ward[province_value][district_value].keys():
                similarity = fuzz.ratio(ward_value, ward)
                if similarity >= 66:
                    return self.reverse_ward[province_value][district_value][ward]
            return None
                
        if ward_value:
            prefix = ['Xã', 'Phường', 'Thị trấn', 'Thị Trấn']
            for pre in prefix:
                if ward_value.startswith(pre):
                    return ward_value
            ward_value = normalize('NFC', ward_value)
            district_value = row['Thành phố/Quận/Huyện/Thị xã']
            province_value = row['Tỉnh/Thành phố']
            return matching(ward_value, district_value, province_value)
        else:
            short_add = row['short_address']
            if isinstance(short_add, str) and short_add != '':
                if 'xã' in short_add.lower():
                    match_result = re.search(pattern='(xã [\w\s]+)', string=short_add.lower())
                    if match_result:
                        match_result = match_result[0]
                        result_split = match_result.split()
                        result = ' '.join(i.capitalize() for i in result_split)
                        return result
                elif 'phường' in short_add.lower():
                    match_result = re.search(pattern='(phường [\w\s]+)', string=short_add.lower())
                    if match_result:
                        match_result = match_result[0]
                        result_split = match_result.split()
                        result = ' '.join(i.capitalize() for i in result_split)
                        return result
                elif 'thị trấn' in short_add.lower():
                    match_result = re.search(pattern='(thị trấn [\w\s]+)', string=short_add.lower())
                    if match_result:
                        match_result = match_result[0]
                        result_split = match_result.split()
                        result = ' '.join(i.capitalize() for i in result_split)
                        return result
                else:
                    short_add_list = row['short_address'].split(',')
                    if len(short_add_list) >= 3:
                        new_province_val = row['Tỉnh/Thành phố']
                        new_ward_val = normalize('NFC',short_add_list[-3].strip())
                        new_district_val = row['Thành phố/Quận/Huyện/Thị xã']
                        return matching(new_ward_val, new_district_val, new_province_val)

            else:
                return None
            
        return None  

## Code Onehousing cleaner

In [ ]:
from Onehousing.orchestrator import (
    scrape_onehousing_urls as oh_scrape_urls, 
    scrape_onehousing_details as oh_scrape_details
)

standardizer = AddressStandardizer(
            PROVINCES_SQL_FILE, DISTRICTS_SQL_FILE, 
            WARDS_SQL_FILE, STREETS_SQL_FILE
        )

class OneHousingDataCleaner:
    """
    Cleans and transforms raw OneHousing scraped data into standardized format.
    """

    @staticmethod
    def _extract_city(row):
        """Extract and standardize city name."""
        city = row.get('city')
        if pd.notna(city):
           return str(row["city"]).replace("TP.", "Thành phố").replace('T.', 'Tỉnh').strip()
        else:
            city = row.get('listing_title', '')
            try:
                return city.split(',')[-1].replace('TP.', 'Thành phố').replace('T.', 'Tỉnh').strip()
            except:
                return np.nan
        # return np.nan
        # if pd.notna(row.get("city")):
            

        # title = str(row.get("listing_title", ""))
        # match = re.search(r"(TP\.|Thành phố)\s*([^.,\n]+)", title, re.IGNORECASE)

        # return f"Thành phố {match.group(2).strip()}" if match else np.nan

    @staticmethod
    def _extract_district(row):
        """Extract and standardize district name."""
        district = row.get("district")

        if pd.notna(district):
            return str(district).replace("Q.", "Quận").replace("H.", "Huyện").replace("TX.", "Thị xã").strip()

        title = row.get('listing_title', '')
        if pd.notna(title):
            return title.split(",")[-2].replace('TP.').replace('Q.', "Quận").replace('H.', "Huyện").replace('TX.', 'Thị xã').strip()
        return np.nan
        # title = str(row.get("listing_title", ""))
        # match = re.search(r"\b(Q\.|H\.|TX\.)\s*([^.,\n]+)", title, re.IGNORECASE)

        # if match:
        #     prefix, name = match.group(1).upper(), match.group(2).strip()
        #     if prefix == "Q.":
        #         return f"Quận {name}"
        #     if prefix == "H.":
        #         return f"Huyện {name}"
        #     if prefix == "TX.":
        #         return f"Thị xã {name}"

        # return np.nan

    @staticmethod
    def _extract_ward(df):
        """Extract ward/commune information."""
        def extract_row(row):
            full_address = row.get('listing_title', '')
            district = row.get('district', '')

            if pd.isna(full_address) or not isinstance(full_address, str) or \
                    pd.isna(district) or not isinstance(district, str):
                return np.nan

            address_list = full_address.split(",")
            try:
                ward = address_list[-3].replace("X.", "Xã").replace("P.", "Phường").replace("TT.", "Thị trấn").strip()
                return ward
            except:
                return np.nan

        return df.apply(extract_row, axis=1)

    @staticmethod
    def _extract_street_name(series: pd.Series) -> pd.Series:
        """Extract street name from listing title."""
        def extract(text: str):
            if pd.isna(text):
                return np.nan

            patterns = [
                r"(?:Nhà mặt ngõ|Đất nền|Nhà trong ngõ).*?cách\s+(.*?)\s*\d+(?:\.\d+)?m",
                r"(?:Nhà mặt phố|Mặt đường)\s+([^,]+?)\s*,",
                r"Đất nền\s+((?!.*cách)[^,]+?)\s*,"
            ]

            for p in patterns:
                if match := re.search(p, str(text), re.IGNORECASE):
                    return re.sub(r'\s*\(.*\)\s*$', '', match.group(1).strip()).strip()
            return np.nan

        return series.apply(extract)

    @staticmethod
    def _classify_property_type(title: str) -> str:
        """Classify property as street-front or alley."""
        if pd.isna(title):
            return ""

        if "cách" in str(title).lower():
            return "Mặt ngõ"

        return "Mặt phố"

    @staticmethod
    def _convert_price_to_numeric(price_str: str) -> float:
        """Convert price string to numeric value."""
        if pd.isna(price_str):
            return np.nan

        price = str(price_str).lower()

        try:
            val_str = price.replace(',', '.').strip()
            if 'tỷ' in val_str:
                return float(val_str.replace('tỷ', '').strip()) * 1e9
            if 'triệu' in val_str:
                return float(val_str.replace('triệu', '').strip()) * 1e6
            return float(val_str)
        except (ValueError, AttributeError):
            return np.nan

    @staticmethod
    def _estimate_price(price: float) -> float:
        """Estimate actual price (98% of listed price)."""
        return round(price * 0.98, 2) if pd.notna(price) else np.nan

    @staticmethod
    def _extract_alley_width(row):
        """Extract minimum alley width."""
        for text in [row.get("alley_width"), row.get("property_description")]:
            if pd.notna(text):
                if nums := re.findall(r"(\d+(?:\.\d+)?)", str(text)):
                    try:
                        return min(float(n) for n in nums)
                    except ValueError:
                        continue
        return np.nan

    @staticmethod
    def _extract_front_width(row):
        """Extract front width from features or description."""
        sources = [row.get('features'), row.get('property_description')]
        patterns = [
            r"Hướng mặt tiền\s*:[^;-]+?-\s*(\d+(?:\.\d+)?)\s*m",
            r"Nhà mặt tiền\s+(\d+(?:\.\d+)?)\s*m"
        ]
        for text in sources:
            if pd.notna(text):
                for p in patterns:
                    if match := re.search(p, str(text), re.IGNORECASE):
                        try:
                            return float(match.group(1))
                        except ValueError:
                            pass
        return np.nan

    @staticmethod
    def _extract_number_of_floors(row):
        """Extract total number of floors."""
        title = str(row.get("listing_title", "")).lower()
        if "đất nền" in title:
            return 0.0

        text = str(row.get("features", "")) + " " + str(row.get("property_description", ""))
        floor = re.search(r"Số tầng:\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
        basement = re.search(r"Số tầng hầm:\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
        total_floors = 0.0
        found = False

        if floor:
            total_floors += float(floor.group(1))
            found = True
        if basement:
            total_floors += float(basement.group(1))
            found = True

        return total_floors if found else np.nan

    @staticmethod
    def _extract_land_area(row):
        """Extract land area."""
        text = str(row.get("features", "")) + " " + str(row.get("property_description", ""))
        patterns = [r"Diện tích:\s*(\d+(?:\.\d+)?)", r"diện tích đất thực tế là\s*([\d.]+)m²"]

        for p in patterns:
            if match := re.search(p, text, re.IGNORECASE):
                try:
                    return float(match.group(1))
                except (ValueError, IndexError):
                    pass

        return np.nan

    @staticmethod
    def _extract_distance_to_main_road(row):
        """Extract distance to main road."""
        desc = str(row.get('property_description', ''))
        title = str(row.get('listing_title', ''))

        if 'mặt phố' in title.lower():
            return 0

        patterns = [
            r"khoảng cách ra trục đường chính\s*(\d+(?:\.\d+)?)\s*m",
            r"cách\s+.*?\s+(\d+(?:\.\d+)?)\s*m"
        ]
        text_sources = [desc, title]

        for text, p in zip(text_sources, patterns):
            if match := re.search(p, text, re.IGNORECASE):
                return float(match.group(1))

        return 0

    @staticmethod
    def _extract_number_of_frontages(row):
        """Extract number of frontages."""
        text = row.get("property_description")
        if pd.isna(text):
            return 1

        match = re.search(r"(\d+)\s*mặt tiền", str(text), re.IGNORECASE)
        if match:
            try:
                return int(match.group(1))
            except (ValueError, IndexError):
                pass

        return 1

    @staticmethod
    def _estimate_remaining_quality(row):
        """Estimate remaining quality of construction."""
        title = str(row.get("listing_title", "")).lower()

        if "đất nền" in title:
            return ""
        return 0.85

    @staticmethod
    def _estimate_construction_price(row):
        """Estimate construction price per sqm."""
        text = str(row.get("features", "")) + " " + str(row.get("property_description", ""))
        floor = re.search(r"Số tầng:\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
        basement = re.search(r"Số tầng hầm:\s*(\d+(?:\.\d+)?)", text, re.IGNORECASE)
        total_floors = 0.0

        if "đất nền" in text:
            return ""

        total_floors += float(floor.group(1)) if floor else 0.0
        total_floors += float(basement.group(1)) if basement else 0.0

        if total_floors == 1:
            return 6275876

        if basement and float(basement.group(1)) > 0:
            return 9504604

        if total_floors > 1:
            return 8221171
        
        return np.nan

    @staticmethod
    def clean_onehousing_data(df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply all cleaning transformations to OneHousing raw data.
        
        Args:
            df: DataFrame with raw OneHousing data
            
        Returns:
            Cleaned DataFrame in standardized format
        """
        print("[OneHousing] Starting data cleaning...")
        
        # Rename property_id to ID for consistency
        if 'property_id' in df.columns:
            df = df.copy()
            df.rename(columns={'property_id': 'ID'}, inplace=True)
        
        # Extract and transform all fields
        city = df.apply(OneHousingDataCleaner._extract_city, axis=1)
        district = df.apply(OneHousingDataCleaner._extract_district, axis=1)
        
        # Temporarily add district column for location extraction
        df_temp = df.copy()
        df_temp['district'] = district
        location = OneHousingDataCleaner._extract_ward(df_temp)
        
        street = OneHousingDataCleaner._extract_street_name(df["listing_title"])
        prop_type = df["listing_title"].apply(OneHousingDataCleaner._classify_property_type)
        price = df["total_price"].apply(OneHousingDataCleaner._convert_price_to_numeric)
        est_price = price.apply(OneHousingDataCleaner._estimate_price)
        floors = df.apply(OneHousingDataCleaner._extract_number_of_floors, axis=1)
        num_frontages = df.apply(OneHousingDataCleaner._extract_number_of_frontages, axis=1)
        area = df.apply(OneHousingDataCleaner._extract_land_area, axis=1)
        front_width = df.apply(OneHousingDataCleaner._extract_front_width, axis=1)
        remaining_quality = df.apply(OneHousingDataCleaner._estimate_remaining_quality, axis=1)
        construction_price = df.apply(OneHousingDataCleaner._estimate_construction_price, axis=1)

        with np.errstate(divide='ignore', invalid='ignore'):
            floors_for_calc = floors.fillna(1.0)
            total_area = round((floors_for_calc * area).replace([np.inf, -np.inf], np.nan), 2)
            length = round((area / front_width).replace([np.inf, -np.inf], np.nan), 2)

        cleaned_df = pd.DataFrame({
            "Tỉnh/Thành phố": city,
            "Thành phố/Quận/Huyện/Thị xã": district,
            "Xã/Phường/Thị trấn": location,
            "Đường phố": street,
            "Chi tiết": prop_type,
            "Nguồn thông tin": df["property_url"],
            "Tình trạng giao dịch": "Đang rao bán",
            "Thời điểm giao dịch/rao bán": np.nan,
            "Thông tin liên hệ": np.nan,
            "Giá rao bán/giao dịch": price,
            "Giá ước tính": est_price,
            "Loại đơn giá (đ/m2 hoặc đ/m ngang)": "đ/m2",
            "Đơn giá đất": np.nan,
            "Lợi thế kinh doanh": np.nan,
            "Số tầng công trình": floors,
            "Tổng diện tích sàn": total_area,
            "Đơn giá xây dựng": construction_price,
            "Năm xây dựng": np.nan,
            "Chất lượng còn lại": remaining_quality,
            "Diện tích đất (m2)": area,
            "Kích thước mặt tiền (m)": front_width,
            "Kích thước chiều dài (m)": length,
            "Số mặt tiền tiếp giáp": num_frontages,
            "Hình dạng": "Chữ nhật",
            "Độ rộng ngõ/ngách nhỏ nhất (m)": df.apply(OneHousingDataCleaner._extract_alley_width, axis=1),
            "Khoảng cách tới trục đường chính (m)": df.apply(OneHousingDataCleaner._extract_distance_to_main_road, axis=1),
            "Mục đích sử dụng đất": "Đất ở",
            "Yếu tố khác": df['property_description'],
            "Tọa độ (vĩ độ)": np.nan,
            "Tọa độ (kinh độ)": np.nan,
            "Hình ảnh của bài đăng": df["image_url"]
        })
        
        print("[OneHousing] Data cleaning completed.")
        test_df = cleaned_df[['Tỉnh/Thành phố', 
        'Thành phố/Quận/Huyện/Thị xã', 
        'Xã/Phường/Thị trấn', 
        'Đường phố', 
        'Chi tiết', 
        'Nguồn thông tin', 
        'Tình trạng giao dịch',
        'Thời điểm giao dịch/rao bán',
        'Thông tin liên hệ',
        'Giá rao bán/giao dịch',
        'Giá ước tính',
        'Loại đơn giá (đ/m2 hoặc đ/m ngang)',
        'Đơn giá đất',
        'Lợi thế kinh doanh',
        'Số tầng công trình',
        'Tổng diện tích sàn',
        'Đơn giá xây dựng',
        'Năm xây dựng',
        "Chất lượng còn lại",
        "Diện tích đất (m2)",
        "Kích thước mặt tiền (m)",
        "Kích thước chiều dài (m)",
        "Số mặt tiền tiếp giáp",
        "Hình dạng",
        "Độ rộng ngõ/ngách nhỏ nhất (m)",
        "Khoảng cách tới trục đường chính (m)",
        "Mục đích sử dụng đất",
        "Yếu tố khác",
        "Tọa độ (vĩ độ)",
        "Tọa độ (kinh độ)",
        "Hình ảnh của bài đăng"]]
        test_df['listing_title'] = df['listing_title']
        test_df['property_description'] = df['property_description']

        return cleaned_df, test_df


# def process_onehousing_df():
#     """Cleaning logic for OneHousing CSV data."""
#     raw_path = DETAILS_CSV_PATH['Onehousing']
#     if not raw_path.exists():
#         return pd.DataFrame()

#     df_raw = pd.read_csv(raw_path)
#     df = OhCleaner.clean_onehousing_data(df_raw)
    
#     # # Apply standardizer to address columns
#     # df['Tỉnh/Thành phố'] = df['Tỉnh/Thành phố'].apply(standardizer.standardize_province)
#     # df['Thành phố/Quận/Huyện/Thị xã'] = df.apply(standardizer.standardize_district, axis=1)
#     # df['Xã/Phường/Thị trấn'] = df.apply(standardizer.standardize_ward, axis=1)
    
#     # Missing coordinates in OH
#     if 'latitude' not in df.columns: df['latitude'] = np.nan
#     if 'longitude' not in df.columns: df['longitude'] = np.nan

#     # Rename to standardized schema
#     oh_final = df.rename(columns={v: k for k, v in FINAL_SCHEMA.items() if v in df.columns})
#     oh_final = oh_final[list(FINAL_SCHEMA.keys())]

#     # Drop NaN and duplicated values
#     na = [
#         'Tỉnh/Thành phố',
#         'Thành phố/Quận/Huyện/Thị xã',
#         'Xã/Phường/Thị trấn',
#         'Đường phố',
#         'Chi tiết',
#         'Nguồn thông tin', 
#         'Thời điểm giao dịch/rao bán',
#         'Giá rao bán/giao dịch',
#         'Giá ước tính',
#         'Số tầng công trình', 
#         'Tổng diện tích sàn', 
#         'Đơn giá xây dựng',
#         'Chất lượng còn lại',
#         'Diện tích đất (m2)',
#         'Kích thước mặt tiền (m)',
#         'Kích thước chiều dài (m)',
#         'Số mặt tiền tiếp giáp',
#         'Hình dạng',
#         'Độ rộng ngõ/ngách nhỏ nhất (m)',
#         'Khoảng cách tới trục đường chính (m)',
#         'Mục đích sử dụng đất',
#         'Tọa độ (vĩ độ)',
#         'Tọa độ (kinh độ)'
#     ]
#     dup = [
#         'Tỉnh/Thành phố', 
#         'Thành phố/Quận/Huyện/Thị xã', 
#         'Xã/Phường/Thị trấn', 
#         'Đường phố', 
#         'Giá rao bán/giao dịch', 
#         'Giá ước tính', 
#         'Số tầng công trình', 
#         'Tổng diện tích sàn', 
#         'Đơn giá xây dựng', 
#         'Chất lượng còn lại', 
#         'Diện tích đất (m2)', 
#         'Kích thước mặt tiền (m)', 
#         'Kích thước chiều dài (m)', 
#         'Số mặt tiền tiếp giáp', 
#         'Hình dạng', 
#         'Độ rộng ngõ/ngách nhỏ nhất (m)', 
#         'Khoảng cách tới trục đường chính (m)', 
#         'Mục đích sử dụng đất'
#     ]

#     old_size = oh_final.shape[0]
#     oh_final.drop_duplicates(subset=dup, inplace=True)
#     oh_final.reset_index(drop=True)
#     print(f'Dropped {old_size - oh_final.shape[0]} duplicated rows for Onehousing.')

#     # old_size = oh_final.shape[0]  
#     # oh_final.dropna(subset=na, inplace=True)
#     # print(f'Dropped {old_size - oh_final.shape[0]} NaN rows for Onehousing.')

#     print(f'Final number of rows for Onehousing: {oh_final.shape[0]}')

    # return oh_final

In [37]:
oh = pd.read_csv('output/onehousing_listings_details.csv', header=None)
oh.columns = ['property_url','listing_title','property_id','total_price','unit_price','alley_width','image_url','city','district','features', 'latitude', 'longitude', 'property_description','scraping_time']
oh.to_csv('output/onehousing_listings_details.csv', index=False)

In [6]:
import pandas as pd
from commons.config import *

oh_raw = pd.read_csv(DETAILS_CSV_PATH['Batdongsan'])
oh_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   url            1000 non-null   object 
 1   title          1000 non-null   object 
 2   id             1000 non-null   int64  
 3   latitude       1000 non-null   float64
 4   longitude      1000 non-null   float64
 5   short_address  1000 non-null   object 
 6   address_parts  1000 non-null   object 
 7   main_info      1000 non-null   object 
 8   description    1000 non-null   object 
 9   other_info     1000 non-null   object 
 10  image_urls     1000 non-null   object 
 11  scraping_time  1000 non-null   int64  
dtypes: float64(2), int64(2), object(8)
memory usage: 93.9+ KB


In [4]:
from Onehousing.cleaning import OneHousingDataCleaner

oh_cleaned = OneHousingDataCleaner.clean_onehousing_data(oh_raw)
oh_cleaned.info()

[OneHousing] Starting data cleaning...
[OneHousing] Data cleaning completed.
<class 'pandas.core.frame.DataFrame'>
Index: 808 entries, 0 to 998
Data columns (total 31 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Tỉnh/Thành phố                        808 non-null    object 
 1   Thành phố/Quận/Huyện/Thị xã           808 non-null    object 
 2   Xã/Phường/Thị trấn                    808 non-null    object 
 3   Đường phố                             808 non-null    object 
 4   Chi tiết                              808 non-null    object 
 5   Nguồn thông tin                       808 non-null    object 
 6   Tình trạng giao dịch                  808 non-null    object 
 7   Thời điểm giao dịch/rao bán           0 non-null      float64
 8   Thông tin liên hệ                     808 non-null    object 
 9   Giá rao bán/giao dịch                 808 non-null    float64
 10  Giá ước tính  

Onehousing NaN ở Xã/Phường/Thị trấn, Thời điểm giao dịch/rao bán, Thông tin liên hệ, Đơn giá đất, Lợi thế kinh doanh, Số tầng công trình, Năm xây dựng, Kích thước mặt tiền (m), Kích thước chiều dài (m), Độ rộng ngõ/ngách nhỏ nhất (m), Yếu tố khác, Tọa độ (kinh độ), Tọa độ (vĩ độ)

# Ghép file Batdongsan

In [16]:
import pandas as pd

old_bds = pd.read_excel('output/old_bds.xlsx')
old_bds_not_drop_na = pd.read_excel('output/old_bds_raw.xlsx')
old_bds_raw = pd.read_csv('output/listing_urls.csv')

new_bds = pd.read_csv('output/batdongsan_listing_urls.csv')

In [17]:
df = pd.concat([old_bds_raw, new_bds], axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60068 entries, 0 to 1519
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     60068 non-null  object
dtypes: object(1)
memory usage: 938.6+ KB


In [26]:
yo = pd.read_csv('output/onehousing_listing_urls.csv', header=None)
yo[0].tolist()

['https://onehousing.vn/bds/Nha-mat-ngo-cach-Hoang-Nhu-Tiep-40m--P--Bo-De--Q--Long-Bien--TP--Ha-Noi.UQPHMR',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Doi-Can-50m--P--Doi-Can--Q--Ba-Dinh--TP--Ha-Noi.TPGLJ0',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Ngo-Thi-Nham-50m--P--Ha-Cau--Q--Ha-Dong--TP--Ha-Noi.BUZJOR',
 'https://onehousing.vn/bds/Nha-mat-pho-Khuong-Trung--P--Khuong-Trung--Q--Thanh-Xuan--TP--Ha-Noi.YBOFZL',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Vinh-Tuy-30m--P--Vinh-Tuy--Q--Hai-Ba-Trung--TP--Ha-Noi.GBNVWC',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Dai-Tu-80m--P--Dai-Kim--Q--Hoang-Mai--TP--Ha-Noi.U0NJMC',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Tan-Mai-50m--P--Thinh-Liet--Q--Hoang-Mai--TP--Ha-Noi.C5PRYW',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Le-Van-Luong-50m--P--Trung-Hoa--Q--Cau-Giay--TP--Ha-Noi.BATVNQ',
 'https://onehousing.vn/bds/Nha-mat-ngo-cach-Duong-Dang-Cong-Chat-100m--TT--Yen-Vien--H--Gia-Lam--TP--Ha-Noi.RT8VQ1',
 'https://onehousing.vn/bds/Nh

# Inspect Database

In [1]:
import pandas as pd
import sqlite3

with sqlite3.connect('output/real_estate.db') as conn:
    df = pd.read_sql("SELECT * FROM cleaned", con=conn)

df

,Tỉnh/Thành phố,Thành phố/Quận/Huyện/Thị xã,Xã/Phường/Thị trấn,Đường phố,Chi tiết,Nguồn thông tin,Tình trạng giao dịch,Thời điểm giao dịch/rao bán,Thông tin liên hệ,Giá rao bán/giao dịch,...,Số mặt tiền tiếp giáp,Hình dạng,Độ rộng ngõ/ngách nhỏ nhất (m),Khoảng cách tới trục đường chính (m),Mục đích sử dụng đất,Yếu tố khác,Tọa độ (vĩ độ),Tọa độ (kinh độ),Hình ảnh của bài đăng,Web
0,Thành phố Hồ Chí Minh,Quận 8,Phường 1,Đường Dương Bá Trạc,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,23/12/2025,None,7.500000e+09,...,1,Chữ nhật,3.5,10.0,Đất ở,"Diện tích 58,9m² công nhận đủ,\nNgang 3,9m dài...",10.745337,106.690188,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
1,Thành phố Hồ Chí Minh,Quận Gò Vấp,Phường 9,Đường Lê Văn Thọ,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,25/12/2025,None,6.550000e+09,...,1,Chữ nhật,5.0,10.0,Đất ở,Khu cao tầng hẻm xe hơi ngang hơn 5m hiếm.\nTặ...,10.846724,106.656586,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
2,Thành phố Hà Nội,Quận Tây Hồ,Phường Xuân La,Đường Xuân La,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,23/12/2025,None,2.800000e+10,...,1,Chữ nhật,5.0,10.0,Đất ở,GIÁ 28TỶ - NGOẠI GIAO ĐOÀN - XUÂN LA - TÂY HỒ ...,21.064891,105.806182,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
3,Thành phố Hà Nội,Quận Long Biên,Phường Phúc Lợi,Đường Phúc Lợi,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,28/12/2025,None,6.700000e+09,...,2,Chữ nhật,2.5,15.0,Đất ở,"Nhà lô góc 2 mặt thoáng, sáng mát tự nhiên. Di...",21.038674,105.934373,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
4,Thành phố Hà Nội,Quận Tây Hồ,Phường Phú Thượng,Đường Phú Thượng,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,30/12/2025,None,1.180000e+10,...,1,Chữ nhật,4.5,100.0,Đất ở,RẺ NHẤT KHU VỰC. CẦN BÁN NHÀ DÂN XÂY 3T 60M2. ...,21.083858,105.808483,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6892,Thành phố Hà Nội,Quận Nam Từ Liêm,Phường Cầu Diễn,Hồ Tùng Mậu,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Ho-...,Đang giao dịch,16/01/2026,None,8.100000e+09,...,1,Chữ nhật,2.0,200.0,Đất ở,None,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6893,Thành phố Hà Nội,Quận Long Biên,Phường Bồ Đề,Bồ Đề,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Bo-...,Đang giao dịch,16/01/2026,None,1.750000e+10,...,1,Chữ nhật,4.0,20.0,Đất ở,None,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6894,Thành phố Hà Nội,Huyện Gia Lâm,Thị trấn Yên Viên,Đường Hà Huy Tập,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Duo...,Đang giao dịch,16/01/2026,None,3.650000e+09,...,1,Chữ nhật,2.0,300.0,Đất ở,None,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6895,Thành phố Hà Nội,Quận Long Biên,Phường Ngọc Thụy,Ngọc Thụy,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Ngo...,Đang giao dịch,16/01/2026,None,8.600000e+09,...,1,Chữ nhật,2.0,200.0,Đất ở,None,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6897 entries, 0 to 6896
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Tỉnh/Thành phố                        6897 non-null   object 
 1   Thành phố/Quận/Huyện/Thị xã           6897 non-null   object 
 2   Xã/Phường/Thị trấn                    6897 non-null   object 
 3   Đường phố                             6897 non-null   object 
 4   Chi tiết                              6897 non-null   object 
 5   Nguồn thông tin                       6897 non-null   object 
 6   Tình trạng giao dịch                  6897 non-null   object 
 7   Thời điểm giao dịch/rao bán           6897 non-null   object 
 8   Thông tin liên hệ                     0 non-null      object 
 9   Giá rao bán/giao dịch                 6897 non-null   float64
 10  Giá ước tính                          6897 non-null   int64  
 11  Loại đơn giá (đ/m

In [4]:
df[df['Web'] == 'Onehousing']['Thời điểm giao dịch/rao bán'].value_counts()

Thời điểm giao dịch/rao bán
16/01/2026    809
Name: count, dtype: int64

In [2]:
import pandas as pd
df_csv = pd.read_csv('output/real_estate_cleaned.csv')
df_csv

,Tỉnh/Thành phố,Thành phố/Quận/Huyện/Thị xã,Xã/Phường/Thị trấn,Đường phố,Chi tiết,Nguồn thông tin,Tình trạng giao dịch,Thời điểm giao dịch/rao bán,Thông tin liên hệ,Giá rao bán/giao dịch,...,Số mặt tiền tiếp giáp,Hình dạng,Độ rộng ngõ/ngách nhỏ nhất (m),Khoảng cách tới trục đường chính (m),Mục đích sử dụng đất,Yếu tố khác,Tọa độ (vĩ độ),Tọa độ (kinh độ),Hình ảnh của bài đăng,Web
0,Thành phố Hồ Chí Minh,Quận 8,Phường 1,Đường Dương Bá Trạc,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,23/12/2025,NaN,7.500000e+09,...,1,Chữ nhật,3.5,10.0,Đất ở,"Diện tích 58,9m² công nhận đủ,\nNgang 3,9m dài...",10.745337,106.690188,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
1,Thành phố Hồ Chí Minh,Quận Gò Vấp,Phường 9,Đường Lê Văn Thọ,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,25/12/2025,NaN,6.550000e+09,...,1,Chữ nhật,5.0,10.0,Đất ở,Khu cao tầng hẻm xe hơi ngang hơn 5m hiếm.\nTặ...,10.846724,106.656586,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
2,Thành phố Hà Nội,Quận Tây Hồ,Phường Xuân La,Đường Xuân La,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,23/12/2025,NaN,2.800000e+10,...,1,Chữ nhật,5.0,10.0,Đất ở,GIÁ 28TỶ - NGOẠI GIAO ĐOÀN - XUÂN LA - TÂY HỒ ...,21.064891,105.806182,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
3,Thành phố Hà Nội,Quận Long Biên,Phường Phúc Lợi,Đường Phúc Lợi,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,28/12/2025,NaN,6.700000e+09,...,2,Chữ nhật,2.5,15.0,Đất ở,"Nhà lô góc 2 mặt thoáng, sáng mát tự nhiên. Di...",21.038674,105.934373,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
4,Thành phố Hà Nội,Quận Tây Hồ,Phường Phú Thượng,Đường Phú Thượng,Mặt ngõ,https://batdongsan.com.vn/ban-nha-rieng-duong-...,Đang rao bán,30/12/2025,NaN,1.180000e+10,...,1,Chữ nhật,4.5,100.0,Đất ở,RẺ NHẤT KHU VỰC. CẦN BÁN NHÀ DÂN XÂY 3T 60M2. ...,21.083858,105.808483,"[""https://file4.batdongsan.com.vn/crop/600x315...",Batdongsan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6892,Thành phố Hà Nội,Quận Nam Từ Liêm,Phường Cầu Diễn,Hồ Tùng Mậu,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Ho-...,Đang giao dịch,NaN,NaN,8.100000e+09,...,1,Chữ nhật,2.0,200.0,Đất ở,NaN,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6893,Thành phố Hà Nội,Quận Long Biên,Phường Bồ Đề,Bồ Đề,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Bo-...,Đang giao dịch,NaN,NaN,1.750000e+10,...,1,Chữ nhật,4.0,20.0,Đất ở,NaN,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6894,Thành phố Hà Nội,Huyện Gia Lâm,Thị trấn Yên Viên,Đường Hà Huy Tập,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Duo...,Đang giao dịch,NaN,NaN,3.650000e+09,...,1,Chữ nhật,2.0,300.0,Đất ở,NaN,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing
6895,Thành phố Hà Nội,Quận Long Biên,Phường Ngọc Thụy,Ngọc Thụy,Mặt ngõ,https://onehousing.vn/bds/Nha-mat-ngo-cach-Ngo...,Đang giao dịch,NaN,NaN,8.600000e+09,...,1,Chữ nhật,2.0,200.0,Đất ở,NaN,21.001010,105.866667,https://rs.onehousing.vn/fast/320x0/filters:qu...,Onehousing


In [8]:
for i in range(df_csv.shape[0]):
    if df.iloc[i]['Nguồn thông tin'] != df_csv.iloc[i]['Nguồn thông tin']:
        print('Different')
    else:
        print('Same')

Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same
Same


In [4]:
df_csv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6897 entries, 0 to 6896
Data columns (total 32 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Tỉnh/Thành phố                        6897 non-null   object 
 1   Thành phố/Quận/Huyện/Thị xã           6897 non-null   object 
 2   Xã/Phường/Thị trấn                    6897 non-null   object 
 3   Đường phố                             6897 non-null   object 
 4   Chi tiết                              6897 non-null   object 
 5   Nguồn thông tin                       6897 non-null   object 
 6   Tình trạng giao dịch                  6897 non-null   object 
 7   Thời điểm giao dịch/rao bán           6088 non-null   object 
 8   Thông tin liên hệ                     0 non-null      float64
 9   Giá rao bán/giao dịch                 6897 non-null   float64
 10  Giá ước tính                          6897 non-null   float64
 11  Loại đơn giá (đ/m

In [7]:
df_csv[df_csv['Yếu tố khác'].isna()]['Web'].value_counts()

Web
Onehousing    809
Batdongsan      2
Name: count, dtype: int64

In [ ]:
from datetime import datetime

print(datetime.now().strftime("%d/%m/%Y"))

16/01/2026


# Test add database

In [1]:
import sqlite3
import pandas as pd
import numpy as np

with sqlite3.connect('output/real_estate.db') as conn:
    bds_raw_old = pd.read_sql("SELECT * FROM bds_raw", con=conn)
    oh_raw_old = pd.read_sql("SELECT * FROM onehousing_raw", con = conn)
    df_old = pd.read_sql("SELECT * FROM cleaned", con=conn)

print(f'BDS RAW: {bds_raw_old.shape}')
print(f'OH RAW: {oh_raw_old.shape}')
print(f'CLEANED RAW: {df_old.shape}')

BDS RAW: (58152, 12)
OH RAW: (869, 13)
CLEANED RAW: (6911, 32)


In [4]:
df['Chi tiết'].unique()

array(['Mặt ngõ', 'Mặt phố', '447', '96', 'Phú Hồng Thịnh 8',
       '925/15 Đường Âu Cơ', 'Ngõ 19', '30/5', 'Số 4 ngõ 5', 'Mỹ Đình 1',
       'Hẻm 413', '214', '3/2', 'Ngõ 17x', 'Ngõ 205', '1/', 'Ngõ 187',
       '7/3', '577 Đường Phạm Văn Bạch', 'Khu đô thị Vĩnh Điềm Trung',
       'Thôn 4', 'Hẻm 38', 'Ngõ 213', 'Ngõ 266', '207 Đường Ngọc Hồi',
       '94/6', 'Ngõ 126', '685', '1/46/ Đặng Thùy Trâm', 'Ngõ 180',
       'Hẻm 1/', '55 Phố Chính Kinh', 'Dự án Green Riverside', 'Hẻm 430',
       'Số 8/233', '45 Đường Liên Khu 1-6', '778', 'Số 50', 'Hẻm 14',
       '11/5', 'Ngõ 20', 'Ngõ 31', 'Hẻm 637', 'Dự án Sài Gòn Mới',
       '86/26', 'KDC Phú Hồng Thịnh 10', 'Hẻm 1056', 'Hẻm 222', '47',
       'Dự án KDC Apec City Center', '56', 'Ngõ 135', 'Hẻm 1x5', 'Ngõ 25',
       'Hẻm 353', 'Ngõ 521', '681/ Đường Âu Cơ', '1K', '111 Đường 9',
       '93 Lò Đúc - Kinh Đô Tower', 'Khu A42',
       'Lô 26BC Đường Lê Hồng Phong', 'Ngõ 147', '1632', '276/3/59',
       'Ngõ 131', '69/40 Đường Nguyễn Côn

In [15]:
with sqlite3.connect('output/real_estate.db') as conn:
    cols = 'property_id, property_url, listing_title, total_price, unit_price, city, district, alley_width, features, latitude, longitude, image_url'
    sql = f"""SELECT {cols}, COUNT(*) 
             FROM onehousing_raw
             GROUP BY {cols}
             HAVING COUNT(*) > 1
             """
    df = pd.read_sql(sql, con=conn)
    print(df.shape)

(869, 13)


In [7]:
with sqlite3.connect('output/real_estate.db') as conn:
    bds_raw = pd.read_sql("SELECT * FROM bds_raw", con=conn)
    oh_raw = pd.read_sql("SELECT * FROM onehousing_raw", con = conn)
    df = pd.read_sql("SELECT * FROM cleaned", con=conn)

print(f'BDS RAW: {bds_raw.shape}')
print(f'OH RAW: {oh_raw.shape}')
print(f'CLEANED RAW: {df.shape}')

BDS RAW: (114251, 12)
OH RAW: (869, 13)
CLEANED RAW: (6911, 32)


# Test newly scraped data

In [2]:
import sqlite3
import pandas as pd
import numpy as np

with sqlite3.connect('output/real_estate.db') as conn:
    bds_raw_old = pd.read_sql("SELECT * FROM bds_raw", con=conn)
    oh_raw_old = pd.read_sql("SELECT * FROM onehousing_raw", con = conn)
    df_old = pd.read_sql("SELECT * FROM cleaned", con=conn)

print(f'BDS RAW: {bds_raw_old.shape}')
print(f'OH RAW: {oh_raw_old.shape}')
print(f'CLEANED RAW: {df_old.shape}')

BDS RAW: (114351, 12)
OH RAW: (969, 13)
CLEANED RAW: (6997, 32)
